In [8]:
import pymongo
import os
import hashlib
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L12-v2')
cadena_conexion = "mongodb://localhost:30001,localhost:30002,localhost:30003/?replicaSet=my-replicaset"


class Discurso:
    def __init__(self, texto):
        self.texto = texto
        self.id = self.hash()
        self.embedding = self.generate_embedding()

    def hash(self):
        time = str(os.times())
        hash_object = hashlib.sha256(self.texto.encode('utf-8') + time.encode('utf-8'))
        return hash_object.hexdigest()

    def generate_embedding(self):
        embedding = model.encode(self.texto)
        return embedding.tolist()

    def to_dict(self):
        return {
            "_id": self.id,
            "texto": self.texto,
            "embedding": self.embedding
        }

def connect():
    try:
        client = pymongo.MongoClient(cadena_conexion, serverSelectionTimeoutMS=5000)
        client.server_info()
        print("Conectado a la base de datos")
        return client
    except Exception as e:
        print("Error conectandose a la base de datos:", e)
        try:
            port = [30001, 30002, 30003]
            for p in port:
                try:
                    client = pymongo.MongoClient(f"mongodb://localhost:{p}/?directConnection=true", serverSelectionTimeoutMS=5000)
                    client.admin.command("ismaster")
                    result = client.admin.command("hello")
                    if result.get("isWritablePrimary", False):
                        print(f"Conectado a la base de datos (conexión directa al primario en el puerto {p})")
                        return client
                except:
                    continue
            return None
        except Exception as e:
            print("Error conectandose a la base de datos (conexión directa al primario):", e)
            return None


In [9]:
client = connect()

try:
    db = client["Politica"]
    collection = db["Discursos"]
    print("Base de datos y colección accedidas con éxito")

    try:
        status = client.admin.command("replSetGetStatus")
        print("Estado del replica set: OK")
    except Exception as e:
        print("Nota: Conectado pero no se puede verificar estado del replica set")
        print("Esto es normal con conexión directa")

    collection.delete_many({})
    print("Todos los documentos han sido eliminados de la colección 'Discursos'.")

    test_doc = Discurso("Documento de prueba")
    collection.insert_one(test_doc.to_dict())
    print("Documento de prueba insertado con éxito.")

    doc = collection.find_one({"_id": test_doc.id})
    if doc:
        print("Documento recuperado con éxito:", doc)
        collection.delete_one({"_id": test_doc.id})
        print("Documento de prueba eliminado con éxito.")
    else:
        print("No se encontró el documento de prueba.")
except Exception as e:
    print("An error occurred:", e)


Conectado a la base de datos
Base de datos y colección accedidas con éxito
Estado del replica set: OK
Todos los documentos han sido eliminados de la colección 'Discursos'.
Documento de prueba insertado con éxito.
Documento recuperado con éxito: {'_id': '79b9e4b83829b11337f0c70e6d798cbb20a3159e8316f29ff757e52efd963e87', 'texto': 'Documento de prueba', 'embedding': [-0.07407179474830627, 0.12956391274929047, -0.06822466105222702, 0.036835554987192154, -0.0011505878064781427, 0.06423676013946533, -0.05256487429141998, 0.10554928332567215, 0.0422363355755806, 0.048714715987443924, 0.05875246599316597, -0.018725819885730743, -0.03310951963067055, 0.023136019706726074, -0.02589540183544159, -0.018097514286637306, 0.0783284604549408, 0.07590683549642563, 0.015536190941929817, 0.053180448710918427, 0.09780590236186981, 0.05147955194115639, 0.013880561105906963, 0.004195868503302336, -0.026801180094480515, -0.01301565021276474, -0.006859882269054651, 0.1147848591208458, -0.02219660021364689, -0

In [10]:
ruta = "./DiscursosOriginales"

documentos = []
documentos_duplicados = []

for archivo in os.listdir(ruta):
    if archivo.endswith(".txt"):
        with open(os.path.join(ruta, archivo), 'r', encoding='utf-8') as f:
            texto = f.read()
            discurso = Discurso(texto)

            if collection.find_one({"_id": discurso.id}):
                documentos_duplicados.append(archivo)
                print(f"⚠ Documento duplicado omitido: {archivo}")
            else:
                documentos.append(discurso.to_dict())


if documentos:
    collection.insert_many(documentos)
    print(f"{len(documentos)} documentos insertados en la colección 'Discursos'.")
else:
    print("No se encontraron documentos para insertar.")


680 documentos insertados en la colección 'Discursos'.


In [11]:
lista_ids = []
lista_textos = []
lista_embeddings = []

try:
    cursor = collection.find({}, {"_id": 1, "embedding": 1, "texto": 1})
    for doc in cursor:
        lista_ids.append(doc["_id"])
        lista_textos.append(doc["texto"])
        lista_embeddings.append(doc["embedding"])
    
    if lista_embeddings:
        matriz_embeddings = np.array(lista_embeddings)
        print(f"Se recuperaron {len(lista_ids)} documentos de la colección 'Discursos'.")
    else:
        print("No se recuperaron embeddings de la colección 'Discursos'.")

except Exception as e:
    print("Error al recuperar documentos de la colección:", e)
    cursor = []

Se recuperaron 680 documentos de la colección 'Discursos'.


In [12]:
def encontrar_mejor_fragmento(textoCompleto, textoConsulta):
    embeddingConsulta = model.encode(textoConsulta).reshape(1, -1)

    fragmentos = textoCompleto.split('. ')
    mejores_fragmentos = []
    for fragmento in fragmentos:
        embeddingFragmento = model.encode(fragmento).reshape(1, -1)
        similitud = cosine_similarity(embeddingConsulta, embeddingFragmento)[0][0]
        mejores_fragmentos.append((fragmento, similitud))

    mejores_fragmentos.sort(key=lambda x: x[1], reverse=True)
    return mejores_fragmentos[0][0] if mejores_fragmentos else None

def buscar_discursos_similares(textoConsulta, top_k=5):
    embeddingConsulta = model.encode(textoConsulta).reshape(1, -1)

    similitudes = cosine_similarity(embeddingConsulta, matriz_embeddings)[0]
    indices_top_k = np.argsort(similitudes)[-top_k:][::-1]

    resultados = []
    for idx in indices_top_k:
        id_doc = lista_ids[idx]
        texto_doc = lista_textos[idx]
        mejor_fragmento = encontrar_mejor_fragmento(texto_doc, textoConsulta)
        resultados.append((id_doc, mejor_fragmento, similitudes[idx]))

    for res in resultados:
        print("\n-------------------------")
        print(f"\n  ID: {res[0]}")
        print(f"\n  SCORE: {res[2]:.4f}")
        print("\n  MEJOR FRAGMENTO: ", res[1])
    return resultados


In [13]:
buscar_discursos_similares("transporte público y calidad de vida")


-------------------------

  ID: 9c8c91b61577e479b4291a08d87bbc8d5bb6531a95e131f60ddec82fdb7e31d1

  SCORE: 0.6197

  MEJOR FRAGMENTO:  Muy buenos días:

 

Hoy es un día muy especial para todos los chilenos que usan o van a usar el transporte público.

 

Nosotros lo dijimos siempre: un país civilizado necesita un sistema de transporte público de calidad, seguro y que además sea económico, limpio y sustentable. 

 

Y a eso apunta nuestro proyecto de reemplazar el tristemente célebre Transantiago, por un nuevo sistema de transporte público que hemos denominado “Transporte Tercer Milenio”.

 

¿En qué consiste esta revolución, que no es solamente en la Región Metropolitana, es en todas las regiones de Chile? 

 

Darles a todos nuestros ciudadanos un sistema de transporte público que sea de calidad, que sea económico, que sea limpio, que garantice los recorridos y los horarios, y que permita a la gente poder disfrutar los tiempos que deba pasar en el transporte público. A eso apunta e

[('9c8c91b61577e479b4291a08d87bbc8d5bb6531a95e131f60ddec82fdb7e31d1',
  'Muy buenos días:\n\n\xa0\n\nHoy es un día muy especial para todos los chilenos que usan o van a usar el transporte público.\n\n\xa0\n\nNosotros lo dijimos siempre: un país civilizado necesita un sistema de transporte público de calidad, seguro y que además sea económico, limpio y sustentable.\xa0\n\n\xa0\n\nY a eso apunta nuestro proyecto de reemplazar el tristemente célebre Transantiago, por un nuevo sistema de transporte público que hemos denominado “Transporte Tercer Milenio”.\n\n\xa0\n\n¿En qué consiste esta revolución, que no es solamente en la Región Metropolitana, es en todas las regiones de Chile?\xa0\n\n\xa0\n\nDarles a todos nuestros ciudadanos un sistema de transporte público que sea de calidad, que sea económico, que sea limpio, que garantice los recorridos y los horarios, y que permita a la gente poder disfrutar los tiempos que deba pasar en el transporte público.\xa0A eso apunta el “Transporte Tercer

In [14]:
import json

def exportarJSON():
    client = pymongo.MongoClient(
        "mongodb://localhost:30001,localhost:30002,localhost:30003/?replicaSet=my-replicaset"
    )
    
    collection = client["Politica"]["Discursos"]
    documentos = list(collection.find({}))
    
    nombre_archivo = "discursos-final-202473522-5-202473509-8.json"
    
    with open(nombre_archivo, 'w', encoding='utf-8') as f:
        json.dump(documentos, f, ensure_ascii=False, indent=2, default=str)
    
    print(f"{len(documentos)} documentos exportados a {nombre_archivo}")
    
    client.close()

exportarJSON()

680 documentos exportados a discursos-final-202473522-5-202473509-8.json
